In [18]:
# -----######-----###### UNIVERSAL TAB-DELIM IMPORT FIXER -----######-----######
import pandas as pd
import chardet
from collections import Counter

def _readfix_2304_txt_GET_df_autoclean(path_txt):
    """
    Reads a tab-delimited text file with inconsistent columns.
    Automatically:
    - Detects encoding
    - Finds dominant column count
    - Truncates or pads rows
    - Returns clean DataFrame
    """
    # Detect encoding
    with open(path_txt, 'rb') as f:
        raw_data = f.read()
        encoding = chardet.detect(raw_data)['encoding']

    lines = raw_data.decode(encoding).splitlines()

    # Split lines and count how many fields in each
    split_lines = [line.strip().split('\t') for line in lines]
    col_lengths = [len(row) for row in split_lines]
    count_freq = Counter(col_lengths)

    # Determine most common column count
    most_common_len = count_freq.most_common(1)[0][0]

    print(f"\n🧠 Most common column count: {most_common_len}")
    print(f"📊 Full column count distribution:")
    for length, freq in sorted(count_freq.items()):
        print(f"{length} columns → {freq} lines")

    # Use first row (or generate header) and fix all rows to match length
    header = split_lines[0]
    if len(header) != most_common_len:
        print("⚠️ Header length mismatch — fixing with auto header")
        header = [f'col_{i}' for i in range(most_common_len)]
        start_idx = 0
    else:
        start_idx = 1

    fixed_rows = []
    for row in split_lines[start_idx:]:
        if len(row) < most_common_len:
            row.extend([''] * (most_common_len - len(row)))
        elif len(row) > most_common_len:
            row = row[:most_common_len]
        fixed_rows.append(row)

    return pd.DataFrame(fixed_rows, columns=header)

# -----######-----######-----######-----######-----
# CORE FUNCTION TO SUM FILE SIZES IN GB
# -----######-----######-----######-----######-----

import os

def _size_2304_i1_GET_total_gb(df):
    """
    Sums the size of all valid files in df['Path'] and returns total in GB (rounded).
    Prints TQM logs.
    """
    total_bytes = 0
    valid_count = 0

    for path in df['Location']:
        if isinstance(path, str) and os.path.isfile(path):
            try:
                total_bytes += os.path.getsize(path)
                valid_count += 1
            except:
                continue

    total_gb = total_bytes / (1024 ** 3)

    print(f"\nTQM ✅ Completed file scan:")
    print(f"   • Valid Files: {valid_count}")
    print(f"   • Total Size: {total_gb:.2f} GB")

    return round(total_gb, 2)

In [19]:
txt_path = "/Users/yerik/Downloads/col_out_arch.txt"
df = _readfix_2304_txt_GET_df_autoclean(txt_path)

print("\n🔎 COLUMN NAMES:")
for i, col in enumerate(df.columns):
    print(f"{i:>2}: '{col}'")
df.head()


🧠 Most common column count: 21
📊 Full column count distribution:
1 columns → 34 lines
4 columns → 1 lines
5 columns → 17 lines
17 columns → 17 lines
18 columns → 1 lines
21 columns → 4826 lines

🔎 COLUMN NAMES:
 0: '#'
 1: 'File Type'
 2: 'Bitrate'
 3: 'BPM'
 4: 'Key'
 5: 'Track Title'
 6: 'Artist'
 7: 'Artwork'
 8: 'Genre'
 9: 'Label'
10: 'Remixer'
11: 'Release Date'
12: 'Year'
13: 'Location'
14: 'Rating'
15: 'File Name'
16: 'Comments'
17: 'Album'
18: 'Bitdepth'
19: 'Time'
20: 'Date Added'


,#,File Type,Bitrate,BPM,Key,Track Title,Artist,Artwork,Genre,Label,...,Release Date,Year,Location,Rating,File Name,Comments,Album,Bitdepth,Time,Date Added
0,1,WAV,1411 kbps,130.00,12A,Ha Dub(2PeKes Vogue Remix),2PeKes,,,,...,,0,/Users/yerik/Music/_0_OLD_SOURCE/_rk_music_35/...,,b9AOl-8_4Bpm_131_0Key_8B_5cnt-rnk_8B3B8B-123oc...,,,16,03:34,2024-08-08
1,2,AIFF,1411 kbps,131.00,6B,Big Moma Tech (Original Mix),Kyle Hall,,Tech House,DistroKid,...,2023-02-17,2023,/Users/yerik/Music/_0_OLD_SOURCE/_rk_music_51/...,,Big_Moma_Tech-BY-Kyle_Hall_(Original_Mix)_KEY_...,,Baci Ballers EP,16,05:39,2023-10-17
2,3,MP3,192 kbps,131.00,6A,Din Daa Daa (Clean),Baltimore Club,,B-More,,...,,0,/Users/yerik/Music/_0_OLD_SOURCE/_rk_music_130...,*,Din Daa Daa (Clean).mp3,,Bmore,16,03:55,2023-08-15
3,4,MP3,320 kbps,84.00,11B,Freak Hoe,Future,,Rap,Epic,...,,2015,/Users/yerik/Music/_0_OLD_SOURCE/_rk_music_133...,,07. Freak Hoe.mp3,,Dirty Sprite 2 (DS2),16,02:54,2023-08-15
4,5,MP3,320 kbps,127.00,1A,La Cocina Del Cabron (Original Mix),"Lee Van Dowski, Glimpse",,Electronica,Global Underground,...,2010-01-31,2010,/Users/yerik/Music/_0_OLD_SOURCE/_rk_music_193...,,"La_Cocina_Del_Cabron-BY-Lee_Van_Dowski,_Glimps...",,Global Underground #38: Carl Cox - Black Rock ...,16,07:11,2024-09-25


# copy files into a new folder 

In [20]:
total_gb = _size_2304_i1_GET_total_gb(df)


TQM ✅ Completed file scan:
   • Valid Files: 4841
   • Total Size: 83.50 GB


In [27]:
# -----######-----###### FIXED COPY LOOP: CORRECT TUPLE UNPACKING -----######-----######
import os
import shutil
import pandas as pd
from tqdm import tqdm

def _fileops_2404_chunkedcopy_GET_genre_sorted_chunks(df, dest_base_path):
    """
    Sorts df by Genre, copies files in chunks of 50 to numbered folders with no filename collisions.
    Adds 'Copied_To' column and saves a CSV log. TQM bar active. No overwrite, all copied.
    """
    tqdm.pandas()
    os.makedirs(dest_base_path, exist_ok=True)
    df = df.copy()

    df = df.sort_values(by='Genre').reset_index(drop=True)
    df['file_name'] = df['Location'].apply(os.path.basename)
    
    copied_to = [None] * len(df)
    folder_idx = 1
    chunk_files = set()
    current_chunk = []

    tq_bar = tqdm(total=len(df), desc="💽 TQM COPY FLOW", ncols=100)

    for idx, row in df.iterrows():
        src = row['Location']
        fname = row['file_name']

        # Start new chunk if needed
        if fname in chunk_files or len(current_chunk) >= 50:
            dest_folder = os.path.join(dest_base_path, f"_24_ARCH_rk_{folder_idx}")
            os.makedirs(dest_folder, exist_ok=True)
            for path, name, i in current_chunk:
                dest_path = os.path.join(dest_folder, name)
                try:
                    shutil.copy2(path, dest_path)
                    copied_to[i] = dest_path
                    tq_bar.update(1)
                except Exception as e:
                    tqdm.write(f"❌ ERROR copying {path} → {dest_path} | {e}")
            tqdm.write(f"📦 Finished chunk: _24_ARCH_rk_{folder_idx} ({len(current_chunk)} files)")

            # Reset chunk
            folder_idx += 1
            chunk_files = set()
            current_chunk = []

        chunk_files.add(fname)
        current_chunk.append((src, fname, idx))

    # Final chunk flush
    if current_chunk:
        dest_folder = os.path.join(dest_base_path, f"_24_ARCH_rk_{folder_idx}")
        os.makedirs(dest_folder, exist_ok=True)
        for path, name, i in current_chunk:
            dest_path = os.path.join(dest_folder, name)
            try:
                shutil.copy2(path, dest_path)
                copied_to[i] = dest_path
                tq_bar.update(1)
            except Exception as e:
                tqdm.write(f"❌ ERROR copying {path} → {dest_path} | {e}")
        tqdm.write(f"📦 Final chunk: _24_ARCH_rk_{folder_idx} ({len(current_chunk)} files)")

    tq_bar.close()
    df['Copied_To'] = copied_to

    # Save log
    log_path = os.path.join(dest_base_path, "copied_log.csv")
    df.to_csv(log_path, index=False)
    print(f"\n✅ Done. Log saved at: {log_path}")
    return df


In [ ]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
updated_df = _fileops_2404_chunkedcopy_GET_genre_sorted_chunks(
    df,
    "/Volumes/MY1TB/_24_ARCH_SONGS"
)





💽 TQM COPY FLOW:   0%|                                                    | 0/4895 [00:00<?, ?it/s]


💽 TQM COPY FLOW:   0%|                                          | 1/4895 [00:02<3:33:27,  2.62s/it]


💽 TQM COPY FLOW:   0%|                                          | 2/4895 [00:03<1:55:21,  1.41s/it]


💽 TQM COPY FLOW:   0%|                                          | 3/4895 [00:03<1:25:49,  1.05s/it]


💽 TQM COPY FLOW:   0%|                                          | 4/4895 [00:04<1:02:55,  1.30it/s]


💽 TQM COPY FLOW:   0%|                                            | 5/4895 [00:04<54:05,  1.51it/s]


💽 TQM COPY FLOW:   0%|                                            | 6/4895 [00:05<50:27,  1.61it/s]


💽 TQM COPY FLOW:   0%|                                            | 7/4895 [00:06<59:04,  1.38it/s]


💽 TQM COPY FLOW:   0%|                                          | 8/4895 [00:06<1:00:32,  1.35it/s]


💽 TQM COPY FLOW:   0%|                                            | 9/4895 [00:

📦 Finished chunk: _24_ARCH_rk_1 (50 files)





💽 TQM COPY FLOW:   1%|▍                                          | 51/4895 [00:31<51:13,  1.58it/s]


💽 TQM COPY FLOW:   1%|▍                                          | 52/4895 [00:31<51:31,  1.57it/s]


💽 TQM COPY FLOW:   1%|▍                                          | 53/4895 [00:32<42:32,  1.90it/s]


💽 TQM COPY FLOW:   1%|▍                                          | 54/4895 [00:32<39:34,  2.04it/s]


💽 TQM COPY FLOW:   1%|▍                                          | 55/4895 [00:32<33:52,  2.38it/s]


💽 TQM COPY FLOW:   1%|▍                                          | 56/4895 [00:33<42:15,  1.91it/s]


💽 TQM COPY FLOW:   1%|▌                                          | 57/4895 [00:34<43:57,  1.83it/s]


💽 TQM COPY FLOW:   1%|▌                                          | 58/4895 [00:35<55:29,  1.45it/s]


💽 TQM COPY FLOW:   1%|▌                                          | 59/4895 [00:35<43:16,  1.86it/s]


💽 TQM COPY FLOW:   1%|▌                                          | 60/4895 [00:

❌ ERROR copying  → /Volumes/MY1TB/_24_ARCH_SONGS/_24_ARCH_rk_2/ | [Errno 2] No such file or directory: ''





💽 TQM COPY FLOW:   2%|▊                                          | 95/4895 [00:49<21:40,  3.69it/s]


💽 TQM COPY FLOW:   2%|▊                                          | 96/4895 [00:50<34:40,  2.31it/s]


💽 TQM COPY FLOW:   2%|▊                                          | 97/4895 [00:51<33:45,  2.37it/s]


💽 TQM COPY FLOW:   2%|▊                                          | 98/4895 [00:51<30:18,  2.64it/s]


                                                                                                    

                                                                                              


                                                                                           
💽 TQM COPY FLOW:   2%|▊                                          | 93/4895 [06:49<31:39,  2.53it/s]

💽 TQM COPY FLOW:   0%|                                                    | 0/4895 [01:19<?, ?it/s]


💽 TQM COPY FLOW:   0%|                                                    | 0/4895 [01:25<?, ?i

...




💽 TQM COPY FLOW:  36%|██████████████                         | 1766/4895 [29:16<3:43:29,  4.29s/it]


💽 TQM COPY FLOW:  36%|██████████████                         | 1767/4895 [29:16<2:38:44,  3.04s/it]


💽 TQM COPY FLOW:  36%|██████████████                         | 1769/4895 [29:17<1:35:39,  1.84s/it]


💽 TQM COPY FLOW:  36%|██████████████                         | 1770/4895 [29:17<1:15:04,  1.44s/it]


💽 TQM COPY FLOW:  36%|██████████████▊                          | 1772/4895 [29:18<52:45,  1.01s/it]


💽 TQM COPY FLOW:  36%|██████████████▊                          | 1773/4895 [29:18<47:22,  1.10it/s]


💽 TQM COPY FLOW:  36%|██████████████▊                          | 1774/4895 [29:19<39:51,  1.30it/s]


💽 TQM COPY FLOW:  36%|██████████████▏                        | 1775/4895 [29:22<1:20:14,  1.54s/it]


💽 TQM COPY FLOW:  36%|██████████████▏                        | 1776/4895 [29:23<1:02:34,  1.20s/it]


💽 TQM COPY FLOW:  36%|██████████████▉                          | 1777/4895 [29:

❌ ERROR copying  → /Volumes/MY1TB/_24_ARCH_SONGS/_24_ARCH_rk_57/ | [Errno 2] No such file or directory: ''





                                                                                                    

                                                                                              


                                                                                           
💽 TQM COPY FLOW:   2%|▊                                          | 93/4895 [36:13<31:39,  2.53it/s]

💽 TQM COPY FLOW:   0%|                                                    | 0/4895 [30:42<?, ?it/s]


💽 TQM COPY FLOW:   0%|                                                    | 0/4895 [30:49<?, ?it/s]

📦 Finished chunk: _24_ARCH_rk_57 (50 files)





💽 TQM COPY FLOW:  37%|██████████████▍                        | 1815/4895 [30:18<1:56:20,  2.27s/it]